Third approach

In [1]:
# Download the lightweight 100-dimensional GloVe vectors
!wget http://nlp.stanford.edu/data/glove.6B.zip
# Unzip only the 100d version to save local space
!unzip -q glove.6B.zip glove.6B.100d.txt
print("GloVe vectors downloaded and extracted successfully!")

--2026-06-06 11:42:52--  http://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://nlp.stanford.edu/data/glove.6B.zip [following]
--2026-06-06 11:42:53--  https://nlp.stanford.edu/data/glove.6B.zip
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2026-06-06 11:42:54--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip’

glov

### Define Vocabulary and Data Preprocessing

To resolve the `NameError` where `vocab` was not defined, we'll first define the necessary imports, preprocessing functions, and the `vocab` dictionary before attempting to create the GloVe embedding matrix.

In [3]:
from datasets import load_dataset
import collections
import re

# 1. Download the real dataset
raw_dataset = load_dataset("dair-ai/emotion", trust_remote_code=True)
EMOTION_MAP = {0: 3, 1: 0, 3: 1, 4: 2, 5: 4}
train_filtered = raw_dataset['train'].filter(lambda batch: batch['label'] in EMOTION_MAP)

# 2. Preprocessing helper
def clean_and_tokenize(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    return text.split()

# 3. Compile all unique words into the vocab variable
all_words = []
for text in train_filtered['text']:
    all_words.extend(clean_and_tokenize(text))

word_counts = collections.Counter(all_words)

# 4. Define the missing vocab dictionary
vocab = {"<PAD>": 0, "<UNK>": 1}
for word, count in word_counts.items():
    if count >= 2:
        vocab[word] = len(vocab)

print(f"Success! 'vocab' has been defined with {len(vocab)} unique words.")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'dair-ai/emotion' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'dair-ai/emotion' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse t

README.md:   0%|          | 0.00/9.05k [00:00<?, ?B/s]

split/train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

split/validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

split/test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/16000 [00:00<?, ? examples/s]

Success! 'vocab' has been defined with 6975 unique words.


In [4]:
import numpy as np
import torch
def create_glove_matrix(vocab, glove_path="glove.6B.100d.txt", embedding_dim=100):
    # 1. Load the GloVe file into a temporary dictionary
    glove_embeddings = {}
    with open(glove_path, 'r', encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            vector = np.asarray(values[1:], dtype='float32')
            glove_embeddings[word] = vector

    # 2. Initialize our PyTorch-compatible matrix with zeros
    matrix_size = len(vocab)
    embedding_matrix = np.zeros((matrix_size, embedding_dim), dtype='float32')

    # 3. Fill the matrix with pre-trained weights where matches exist
    for word, idx in vocab.items():
        if word in glove_embeddings:
            embedding_matrix[idx] = glove_embeddings[word]
        else:
            # For <PAD> or unique unknown words, initialize randomly
            embedding_matrix[idx] = np.random.normal(scale=0.6, size=(embedding_dim,))

    return torch.tensor(embedding_matrix)

# Build the weight matrix based on your existing 'vocab' from the previous steps
glove_weights = create_glove_matrix(vocab, embedding_dim=100)
print(f"Pre-trained embedding matrix created with shape: {glove_weights.shape}")

Pre-trained embedding matrix created with shape: torch.Size([6975, 100])


In [6]:
import torch
import torch.nn as nn  # This fixes the error!
class AdvancedLSTMEmotionClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, pretrained_weights):
        super(AdvancedLSTMEmotionClassifier, self).__init__()

        # Load pre-trained GloVe weights into the embedding layer
        self.embedding = nn.Embedding.from_pretrained(pretrained_weights, freeze=False, padding_idx=0)

        # Upgraded to Bidirectional=True
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True,
            num_layers=1
        )

        self.dropout = nn.Dropout(0.3)

        # Because it's bidirectional, the hidden state size doubles (hidden_dim * 2)
        self.fc = nn.Linear(in_features=hidden_dim * 2, out_features=output_dim)

    def forward(self, text):
        embedded = self.embedding(text)
        lstm_out, (h_n, c_n) = self.lstm(embedded)

        # Concatenate the final forward hidden state and backward hidden state
        # h_n shape: [num_layers * num_directions, batch_size, hidden_dim]
        hidden_forward = h_n[-2, :, :]
        hidden_backward = h_n[-1, :, :]
        combined_hidden = torch.cat((hidden_forward, hidden_backward), dim=1)

        combined_hidden = self.dropout(combined_hidden)
        logits = self.fc(combined_hidden)
        return logits

# Re-initialize the model with the GloVe matrix configurations
EMBEDDING_DIM = 100 # Must match GloVe dimension
HIDDEN_DIM = 64
OUTPUT_CLASSES = 5

# Pass the model to device (CPU is perfectly fine here)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = AdvancedLSTMEmotionClassifier(len(vocab), EMBEDDING_DIM, HIDDEN_DIM, OUTPUT_CLASSES, glove_weights).to(device)
print(model)

AdvancedLSTMEmotionClassifier(
  (embedding): Embedding(6975, 100, padding_idx=0)
  (lstm): LSTM(100, 64, batch_first=True, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=128, out_features=5, bias=True)
)


In [7]:
import torch
from torch.utils.data import Dataset, DataLoader

# 1. Define the Dataset Class (Logic for processing text into tensors)
class MassiveEmotionDataset(Dataset):
    def __init__(self, hf_dataset, vocab, emotion_map, max_len=30):
        self.dataset = hf_dataset
        self.vocab = vocab
        self.emotion_map = emotion_map
        self.max_len = max_len

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        tokens = clean_and_tokenize(item['text'])

        # Convert words to indices, using 1 for unknown words
        numerical_seq = [self.vocab.get(token, 1) for token in tokens]

        # Padding/Truncating to a fixed length of 30
        if len(numerical_seq) < self.max_len:
            numerical_seq += [0] * (self.max_len - len(numerical_seq))
        else:
            numerical_seq = numerical_seq[:self.max_len]

        new_label = self.emotion_map[item['label']]
        return torch.tensor(numerical_seq, dtype=torch.long), torch.tensor(new_label, dtype=torch.long)

# 2. Create the train_loader (This is what was missing!)
# We use a batch_size of 64 for a good balance between speed and accuracy
max_sequence_length = 30
train_dataset = MassiveEmotionDataset(train_filtered, vocab, EMOTION_MAP, max_len=max_sequence_length)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

print(f"Success! 'train_loader' is now ready with {len(train_loader)} batches.")

Success! 'train_loader' is now ready with 230 batches.


In [8]:
# Remap targets check: 0: Happiness, 1: Anger, 2: Fear, 3: Sadness, 4: Surprise
# We give Sadness (index 3) a higher penalty multiplier to boost recall
class_weights = torch.tensor([1.0, 1.0, 1.0, 2.0, 1.0]).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=0.002)

epochs = 10
model.train()

print("Training advanced model...")
for epoch in range(epochs):
    epoch_loss = 0
    correct = 0
    total = 0

    for batch_text, batch_labels in train_loader:
        batch_text, batch_labels = batch_text.to(device), batch_labels.to(device)

        optimizer.zero_grad()
        predictions = model(batch_text)
        loss = criterion(predictions, batch_labels)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        _, predicted_classes = torch.max(predictions, dim=1)
        correct += (predicted_classes == batch_labels).sum().item()
        total += batch_labels.size(0)

    print(f"Epoch {epoch+1:02d}/{epochs} | Loss: {epoch_loss/len(train_loader):.4f} | Accuracy: {(correct/total)*100:.1f}%")

Training advanced model...
Epoch 01/10 | Loss: 1.0463 | Accuracy: 50.8%
Epoch 02/10 | Loss: 0.3799 | Accuracy: 82.4%
Epoch 03/10 | Loss: 0.1794 | Accuracy: 93.1%
Epoch 04/10 | Loss: 0.1184 | Accuracy: 95.0%
Epoch 05/10 | Loss: 0.0811 | Accuracy: 96.4%
Epoch 06/10 | Loss: 0.0562 | Accuracy: 97.5%
Epoch 07/10 | Loss: 0.0513 | Accuracy: 97.9%
Epoch 08/10 | Loss: 0.0381 | Accuracy: 98.4%
Epoch 09/10 | Loss: 0.0268 | Accuracy: 98.9%
Epoch 10/10 | Loss: 0.0222 | Accuracy: 99.0%


In [14]:
def predict_advanced_emotion(sentence, model, vocab, max_len=30):
    model.eval()
    categories = {0: "Happiness", 1: "Anger", 2: "Fear", 3: "Sadness", 4: "Surprise"}

    tokens = clean_and_tokenize(sentence)
    numerical_seq = [vocab.get(token, 1) for token in tokens]

    if len(numerical_seq) < max_len:
        numerical_seq += [0] * (max_len - len(numerical_seq))
    else:
        numerical_seq = numerical_seq[:max_len]

    input_tensor = torch.tensor(numerical_seq, dtype=torch.long).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(input_tensor)
        probabilities = torch.softmax(logits, dim=1)
        confidence, predicted_idx = torch.max(probabilities, dim=1)

    return categories[predicted_idx.item()], confidence.item()

# Test the exact line that failed before!
test_phrases = [
    " My aunt slapped his son."
     # Checking contrastive signals
]

for p in test_phrases:
    emotion, conf = predict_advanced_emotion(p, model, vocab, max_len=30)
    print(f"Text: '{p}' -> Predicted: {emotion} ({conf*100:.1f}%)")

Text: ' My aunt slapped his son.' -> Predicted: Sadness (72.0%)


### Push to GitHub

This section handles the authentication and pushing of this notebook to a new repository named `Multi-Class_Emotion_Detection`.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
